ㄹㄹㅎㅎㄹㄹ로젝트 루트 확인과 4개 CSV 불러오기

 

35. 프로젝트 루트 설정


In [64]:
# 35. 프로젝트 루트 설정

from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":

    project_root = project_root.parent

data_dir = project_root / "data" / "raw"

print("프로젝트 루트:", project_root)

print("데이터 폴더:", data_dir)

print("데이터 폴더 존재:", data_dir.exists())


프로젝트 루트: c:\dev\ai-data-analysis-repository
데이터 폴더: c:\dev\ai-data-analysis-repository\data\raw
데이터 폴더 존재: True


# 36. pandas와 CSV 불러오기

In [65]:

import pandas as pd

customers = pd.read_csv(data_dir / "customers.csv")
products = pd.read_csv(data_dir / "products.csv")
orders = pd.read_csv(data_dir / "orders.csv")
order_items = pd.read_csv(data_dir / "order_items.csv")

# 37. 기본 구조와 주요 키 확인


In [66]:
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,

}

for name, df in datasets.items():

    print(name, df.shape, df.columns.tolist())

customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


In [67]:

key_checks = {

    "customers.customer_id": customers["customer_id"],
    "products.product_id": products["product_id"],
    "orders.order_id": orders["order_id"],
    "order_items.order_item_id": order_items["order_item_id"],

}

for name, series in key_checks.items():
    print(
        name,
        "결측:", series.isna().sum(),
        "중복:", series.duplicated().sum(),
    )


customers.customer_id 결측: 0 중복: 0
products.product_id 결측: 0 중복: 0
orders.order_id 결측: 0 중복: 0
order_items.order_item_id 결측: 0 중복: 0


Part 10. 컬럼 선택·조건 필터링·정렬

 

38. Series와 DataFrame 선택

In [68]:
city_series = customers["city"]
customer_view = customers[
    ["customer_id", "gender", "age", "city"]
]
print(type(city_series))
print(type(customer_view))
display(customer_view.head())

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


39. 단일 조건 필터링

In [69]:
 
customers_over_30 = customers[
    customers["age"] >= 30
]
print(len(customers), len(customers_over_30))
display(customers_over_30.head())

150 111


,customer_id,name,gender,age,city,signup_date
1,2,김정호,F,32,대구,2025-11-30
2,3,이경수,F,61,성남,2024-07-10
3,4,조영호,F,55,울산,2026-05-11
5,6,김지원,F,32,성남,2026-07-25
6,7,이상현,F,53,인천,2025-01-09


40. 복합 조건 필터링

In [70]:

seoul_over_30 = customers[
    (customers["age"] >= 30)
    & (customers["city"] == "서울")

]

display(seoul_over_30.head())

,customer_id,name,gender,age,city,signup_date
8,9,송지민,M,69,서울,2025-11-16
14,15,장정식,M,69,서울,2026-07-02
29,30,이민재,F,32,서울,2023-08-11
47,48,김예은,F,47,서울,2025-04-29
65,66,김재호,F,39,서울,2025-12-31


In [71]:
seoul_or_busan = customers[
    customers["city"].isin(["서울", "부산"])
]

display(
    seoul_or_busan["city"].value_counts()
)

city
부산    16
서울    15
Name: count, dtype: int64

In [72]:

not_completed = orders[
    ~(orders["order_status"] == "completed")
]
display(
    not_completed["order_status"].value_counts(
        dropna=False
    )
)

order_status
cancelled    64
refunded     52
Name: count, dtype: int64

41. 상품 가격 정렬

In [73]:
expensive_products = (

    products
    .sort_values("price", ascending=False)
    .head(10)
)
display(
    expensive_products[
        [
            "product_id",
            "product_name",
            "category",
            "price",
        ]
    ]

)

,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


Part 11. line_total 생성과 전체 주문 금액 구분

 

42. 작업용 복사본과 파생 컬럼

In [74]:

order_items_work = order_items.copy()
order_items_work["line_total"] = (
    order_items_work["quantity"]
    * order_items_work["unit_price"]

_IncompleteInputError: incomplete input (763737766.py, line 4)

In [ ]:

display(

    order_items_work[

        [

            "order_item_id",

            "order_id",

            "product_id",

            "quantity",

            "unit_price",

            "line_total",

        ]

    ].head()
)

,order_item_id,order_id,product_id,quantity,unit_price,line_total
0,1,1,100,3,102000,306000
1,2,1,87,5,25000,125000
2,3,1,7,3,142000,426000
3,4,1,9,3,193000,579000
4,5,2,72,4,189000,756000


43. 수작업 검증

In [ ]:
sample = order_items_work.iloc[0]

expected = sample["quantity"] * sample["unit_price"]

actual = sample["line_total"]

print("수작업:", expected)

print("파생 컬럼:", actual)

print("일치:", expected == actual)

수작업: 306000
파생 컬럼: 306000
일치: True


In [75]:
#44. 전체 주문상세 금액
all_order_amount = order_items_work["line_total"].sum()
print("전체 주문상세 금액:", all_order_amount)

전체 주문상세 금액: 255610000


In [76]:
#45. 병합용 주문 컬럼 선택

orders_for_merge = orders[
    [
        "order_id",
        "customer_id",
        "order_date",
        "order_status",
    ]

].copy()

print(orders_for_merge.shape)
print(orders_for_merge.head())


(300, 4)
   order_id  customer_id  order_date order_status
0         1          123  2026-06-04    completed
1         2           77  2025-08-20    cancelled
2         3          138  2025-12-17    cancelled
3         4           57  2026-02-27    cancelled
4         5          125  2026-01-18    cancelled


In [77]:
#46. 주문상세와 주문 병합

order_sales = (
    order_items_work
    .merge(
        orders_for_merge,
        on="order_id",
        how="left",
        validate="many_to_one",
        indicator="order_match",
    )
)

In [78]:
#47. 병합 검증

print("병합 전 행 수:", len(order_items_work))
print("병합 후 행 수:", len(order_sales))
display(
    order_sales["order_match"].value_counts(
        dropna=False
    )
)

병합 전 행 수: 764
병합 후 행 수: 764


order_match
both          764
left_only       0
right_only      0
Name: count, dtype: int64

In [80]:
#미매칭 확인:
unmatched_orders = order_sales[
    order_sales["order_match"] != "both"
]
display(unmatched_orders.head())

,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,order_status,order_match


In [81]:
#48. 완료 주문 분석셋
display(
    order_sales["order_status"].value_counts(
        dropna=False
    )
)

order_status
completed    474
cancelled    162
refunded     128
Name: count, dtype: int64

In [82]:

completed_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()

In [86]:
print("완료 주문상세 행:", len(completed_sales))
print("완료 주문 수:",
    completed_sales["order_id"].nunique(),
)

print("완료 주문 고객 수:",
    completed_sales["customer_id"].nunique(),
)

print("완료 주문 매출:",
    completed_sales["line_total"].sum(),
)

완료 주문상세 행: 474
완료 주문 수: 184
완료 주문 고객 수: 100
완료 주문 매출: 148990000


In [87]:
#49. 필요한 상품 정보만 선택

products_for_merge = products[
    ["product_id",
        "product_name",
        "category",]
].copy()
print(products_for_merge.head())

   product_id product_name category
0           1  전자기기 상품 001     전자기기
1           2    도서 상품 002       도서
2           3  전자기기 상품 003     전자기기
3           4  생활용품 상품 004     생활용품
4           5    식품 상품 005       식품


In [88]:
#50. 완료 주문상세와 상품 병합

completed_items = (

    completed_sales
    .merge(
        products_for_merge,
        on="product_id",
        how="left",
        validate="many_to_one",
        indicator="product_match",
    )
)

In [89]:
print(len(completed_sales), len(completed_items))
display(
    completed_items["product_match"].value_counts(
        dropna=False
    )
)

474 474


product_match
both          474
left_only       0
right_only      0
Name: count, dtype: int64

In [90]:
category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
    .sort_values("total_sales", ascending=False)
)
display(category_sales)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
3,스포츠,31743000,85,67,295,100
5,전자기기,26400000,60,44,259,78
2,생활용품,23915000,65,50,272,83
1,뷰티,23383000,65,53,223,76
4,식품,16573000,36,31,133,42
0,도서,16389000,52,46,149,58
6,패션,10587000,33,27,111,37


In [91]:
#52. 카테고리 합계 검증
category_total = category_sales["total_sales"].sum()
completed_total = completed_items["line_total"].sum()
print(category_total)
print(completed_total)
print(category_total == completed_total)

148990000
148990000
True


In [92]:
#53. 상품별 매출

product_sales = (
    completed_items
    .groupby(
        ["product_id", "product_name", "category"],
        as_index=False,
    )
    .agg(
        total_sales=("line_total", "sum"),
        quantity_sold=("quantity", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
    )
    .sort_values("total_sales", ascending=False)
)
display(product_sales.head(10))

,product_id,product_name,category,total_sales,quantity_sold,order_count,customer_count
39,41,스포츠 상품 041,스포츠,5705000,35,12,11
11,12,식품 상품 012,식품,4375000,25,7,7
8,9,스포츠 상품 009,스포츠,3860000,20,6,5
70,72,뷰티 상품 072,뷰티,3780000,20,6,6
69,71,전자기기 상품 071,전자기기,3703000,23,5,5
66,68,스포츠 상품 068,스포츠,3640000,26,8,8
78,81,전자기기 상품 081,전자기기,3630000,22,6,6
10,11,패션 상품 011,패션,3565000,31,7,7
20,22,생활용품 상품 022,생활용품,3248000,29,8,8
86,89,생활용품 상품 089,생활용품,3090000,30,11,11
